# easymlops 表格新 pipe 示例（v0.2.5 / v0.2.6）

源自 kaggle_solutions 离线语料挖掘的高频优胜技术，展示四梯队 pipe 的典型用法：

**第一梯队** SeedAveraging / PseudoLabelPipe / PredictionPostProcess ·
**第二梯队** GroupAggregateFeatures / ClusterFeatures / QuantileClip / LogTransform / CountEncoding / CategoryCross / CorrFilter ·
**第三梯队** NullImportanceFilter / RFEFilter / TextTfidfSvd / OptunaTuner / TargetEncoding(OOF) ·
**第四梯队** TabularMLP(需 torch)

In [1]:
import os, sys
try:
    import easymlops
except ImportError:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
    import easymlops

import numpy as np
import pandas as pd

rng = np.random.RandomState(7)
n = 600
df = pd.DataFrame({
    "city": rng.choice(["bj", "sh", "sz"], n),
    "product": rng.choice(["a", "b", "c", "d"], n),
    "income": np.abs(rng.normal(8000, 2000, n)) + 1,
    "value": rng.normal(100, 20, n),
    "remark": [words[i % 5] for i, words in enumerate(
        [["good service fast", "bad quality slow", "excellent support",
          "terrible experience", "average product"]] * n)],
})
df["y"] = ((df["income"] > df["income"].median()).astype(int) *
           (df["city"] != "bj").astype(int)).values
train, test = df[:480].copy(), df[480:].copy()
print(train.shape, test.shape, "| cols:", list(train.columns))

(480, 6) (120, 6) | cols: ['city', 'product', 'income', 'value', 'remark', 'y']


## 1. 分组聚合 + 频次编码 + 类别交叉

`GroupAggregateFeatures` 把 groupby 统计回填到行（Kaggle 表格赛第一数据增强），
`CategoryCross` 生成类别交叉列（接 TargetEncoding 完成编码），`CountEncoding`
做频次编码。未见组/未见取值自动回退。

In [2]:
from easymlops.table.features import GroupAggregateFeatures, QuantileClip
from easymlops.table.encoding import CategoryCross, CountEncoding

p = easymlops.TablePipeLine()
p.pipe(GroupAggregateFeatures(group_cols=["city"], stats=["mean", "count"])) \
 .pipe(CategoryCross(cols=[["city", "product"]])) \
 .pipe(CountEncoding(cols=["city"], normalize=True))
p.fit(train)
out = p.transform(test.copy())
print("new cols:", sorted(c for c in out.columns if c not in train.columns))
print(out[["income_mean", "income_count", "city_product_cross", "city"]].head(3))

new cols: ['city_product_cross', 'income_count', 'income_mean', 'value_count', 'value_mean', 'y_count', 'y_mean']
     income_mean  income_count city_product_cross      city
480  8163.132663         154.0               sz_a  0.320833
481  8163.132663         154.0               sz_a  0.320833
482  7936.868222         153.0               sh_a  0.318750


## 2. 分位数截断 + 对数变换（抗偏态/异常值）

`QuantileClip` 按训练集分位数自动定界（Winsorize），`LogTransform` 做 log1p。

In [3]:
from easymlops.table.features import LogTransform

income_before = test["income"].copy()
qc = QuantileClip(cols=["income"], lower=0.05, upper=0.95)
qc.fit(train)
clipped = qc.transform(test.copy())
print("clip range: [%.1f, %.1f] (origin [%.1f, %.1f])"
      % (clipped["income"].min(), clipped["income"].max(),
         income_before.min(), income_before.max()))

clip range: [5020.7, 11138.5] (origin [1490.0, 12727.8])


## 3. 相关列去重 CorrFilter

高相关列对中删除与目标相关性更低的一列（提供 y 时）。

In [4]:
from easymlops.table.feature_selection import CorrFilter

cdf = train.copy()
cdf["value_copy"] = cdf["value"]          # 冗余重复列
cf = CorrFilter(threshold=0.95, y=cdf["y"])
cf.fit(cdf)
print("dropped:", cf.dropped_cols_)
print("selected:", cf.selected_cols)

dropped: ['value_copy']
selected: ['city', 'product', 'income', 'value', 'remark', 'y']


## 4. SeedAveraging 多种子平均 + PredictionPostProcess 后处理

同构模型多种子训练取平均（需配 bagging/feature_fraction 使种子生效），
`PredictionPostProcess` 挂管道尾部做 clip/round/二值化。

In [5]:
from easymlops.table.ensemble import SeedAveraging
from easymlops.table.postprocess import PredictionPostProcess
from easymlops.table.regression import LGBMRegression

target_reg = train["value"] * 2
pipeline = easymlops.TablePipeLine()
pipeline.pipe(CountEncoding(cols=["city", "product"])) \
        .pipe(SeedAveraging(
    base_pipe=LGBMRegression(y=target_reg, use_faster_predictor=False,
                             native_init_params={"bagging_fraction": 0.8,
                                                 "bagging_freq": 1,
                                                 "feature_fraction": 0.8}),
    n_seeds=3, seed_start=42)) \
       .pipe(PredictionPostProcess(cols=["pred"], clip=(0, None), round_digits=1))
pipeline.fit(train.drop(columns=["value", "y", "remark"]))
result = pipeline.transform(test.drop(columns=["value", "y", "remark"]).copy())
print("pred (clip+round):", result["pred"].head(3).tolist())

the <class 'easymlops.table.postprocess.PredictionPostProcess'> and <class 'easymlops.table.core.pipe.TablePipeObjectBase'> use same parameter names [['copy_data', 'input_col_names', 'transform_check_max_number_error', 'output_col_names', 'skip_check_transform_value', 'prefix', 'branch_pipes', 'skip_check_transform_type']],please check if conflict,
pred (clip+round): [204.0, 202.8, 178.5]


## 5. PseudoLabelPipe 伪标签半监督

用已训练模型对无标签数据预测，高置信样本打伪标签拼回训练集重训。

In [6]:
from easymlops.table.classification import LGBMClassification
from easymlops.table.ensemble import PseudoLabelPipe

# 数值特征(类别列演示略, 可先 CountEncoding/LabelEncoding)
feats_train = train.drop(columns=["y", "city", "product", "remark"])
feats_test = test.drop(columns=["y", "city", "product", "remark"])
pl = PseudoLabelPipe(
    base_pipe=LGBMClassification(y=train["y"], use_faster_predictor=False),
    confidence=0.7, n_rounds=1)
pl.fit(feats_train, unlabeled=feats_test)
probs = pl.transform(feats_test.copy())
print("prob cols:", list(probs.columns), "| rows:", len(probs),
      "| row sum=1:", bool(np.allclose(probs.sum(axis=1), 1.0, atol=1e-5)))

prob cols: [0, 1] | rows: 120 | row sum=1: True


## 6. TextTfidfSvd 文本特征

自由文本列 → TF-IDF → TruncatedSVD 压缩特征，让 LGBM 消费文本。

In [7]:
from easymlops.table.text_features import TextTfidfSvd

tfs = TextTfidfSvd(cols=["remark"], n_components=3)
tfs.fit(train)
tout = tfs.transform(test.copy())
print("added cols:", [c for c in tout.columns if c.startswith("remark_svd")])
print(tout[["remark_svd_0", "remark_svd_1", "remark_svd_2"]].head(3))

the <class 'easymlops.table.text_features.TextTfidfSvd'> and <class 'easymlops.table.core.pipe.TablePipeObjectBase'> use same parameter names [['prefix']],please check if conflict,
added cols: ['remark_svd_0', 'remark_svd_1', 'remark_svd_2']
     remark_svd_0  remark_svd_1  remark_svd_2
480     -0.291502      0.032796      0.669255
481      0.369582      0.138959      0.727081
482      0.790759     -0.480945     -0.017844


## 7. TabularMLP 表格神经网络（需 torch）

数值列自动标准化 + 类别列自动 Embedding（未见值→id 0）+ MLP；
分类输出概率列，权重随 save/load 持久化。

In [8]:
from easymlops.table.nn import TabularMLPClassification

mlp = TabularMLPClassification(y=train["y"], epochs=10, hidden_dims=(64, 32),
                               random_state=42)
mlp.fit(train.drop(columns=["y", "remark"]))
probs_mlp = mlp.transform(test.drop(columns=["y", "remark"]).copy())
print("prob cols:", list(probs_mlp.columns),
      "| row sum=1:", bool(np.allclose(probs_mlp.sum(axis=1), 1.0, atol=1e-5)))
mlp.save("_demo_mlp.pkl")
mlp_loaded = TabularMLPClassification()
mlp_loaded.load("_demo_mlp.pkl")
probs2 = mlp_loaded.transform(test.drop(columns=["y", "remark"]).copy())
print("save/load consistent:", bool(np.allclose(probs_mlp.values, probs2.values, atol=1e-6)))
os.remove("_demo_mlp.pkl")

the <class 'easymlops.table.nn.NNBase'> and <class 'easymlops.table.core.pipe.TablePipeObjectBase'> use same parameter names [['copy_data', 'input_col_names', 'transform_check_max_number_error', 'output_col_names', 'skip_check_transform_value', 'prefix', 'branch_pipes', 'skip_check_transform_type']],please check if conflict,
prob cols: [0, 1] | row sum=1: True
the <class 'easymlops.table.nn.NNBase'> and <class 'easymlops.table.core.pipe.TablePipeObjectBase'> use same parameter names [['copy_data', 'input_col_names', 'transform_check_max_number_error', 'output_col_names', 'skip_check_transform_value', 'prefix', 'branch_pipes', 'skip_check_transform_type']],please check if conflict,
save/load consistent: True


## 8. 特征筛选：NullImportanceFilter / RFEFilter

空重要性筛选（y 打乱对照）与 LGBM 重要性贪心消除。

In [9]:
from easymlops.table.feature_selection import NullImportanceFilter, RFEFilter

nf = NullImportanceFilter(y=train["y"], n_runs=10, percentile=75,
                          native_init_params={"n_estimators": 60})
nf.fit(train.drop(columns=["y", "remark"]))
print("NullImportance kept:", nf.selected_cols)

rfe = RFEFilter(y=train["y"], n_features_to_select=4, task="classification",
                native_init_params={"n_estimators": 40})
rfe.fit(train.drop(columns=["y", "remark"]))
print("RFE kept:", rfe.selected_cols)

NullImportance kept: ['city', 'product']
RFE kept: ['city', 'product', 'income', 'value']


## 相关测试

- `tests/test_new_pipes.py`（第一梯队 18 项）
- `tests/test_second_pipes.py`（第二梯队 32 项）
- `tests/test_third_pipes.py`（第三梯队 17 项）
- `tests/test_nn_pipes.py`（TabularMLP 12 项）